In [1]:
from simbanator.io.simba import Simulation
from simbanator.analysis.particles import extract_particles
from astropy.io import fits
import numpy as np
# Load your simulation (use the name you configured)
sim = Simulation("cis100")  # Replace XX with your snapshot number

In [2]:
from pathlib import Path
import textwrap
import subprocess

targetfile = '/mnt/home/glorenzon/output/quenching_times/legacy_zsp_sampled_snapshots_with_ids.fits'
with fits.open(targetfile) as hdul:
    data = hdul[1].data
    id_draws = np.asarray(data['ID_AT_SELECTED_SNAP'], dtype=np.int64)
    snaps = np.asarray(data['SNAP_SELECTED'], dtype=np.int64)

# unique (snapshot, galaxy_id) pairs -> one array task per pair
jobs = np.unique(np.column_stack([snaps, id_draws]), axis=0)
if jobs.size == 0:
    raise RuntimeError('No jobs found in FITS table.')

job_dir = Path.cwd() / 'output' / 'slurm' / 'filter_particles'
log_dir = job_dir / 'logs'
job_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

manifest_path = job_dir / 'jobs.tsv'
np.savetxt(
    manifest_path,
    jobs,
    fmt='%d\t%d',
    header='snap\tgalaxy_id',
    comments=''
 )

worker_path = job_dir / 'run_filter_particles_worker.py'
worker_path.write_text(
    textwrap.dedent(
        '''
        import argparse
        import numpy as np
        from simbanator.io.simba import Simulation
        from simbanator.analysis.particles import extract_particles

        def main():
            parser = argparse.ArgumentParser()
            parser.add_argument('--manifest', required=True)
            parser.add_argument('--task-id', type=int, required=True)
            parser.add_argument('--simulation', default='cis100')
            args = parser.parse_args()

            rows = np.loadtxt(args.manifest, dtype=np.int64, skiprows=1)
            if rows.ndim == 1:
                rows = rows[np.newaxis, :]

            if args.task_id < 0 or args.task_id >= len(rows):
                raise IndexError(
                    f'task_id={args.task_id} is out of range for {len(rows)} jobs'
                )

            snap, galaxy_id = rows[args.task_id]
            sim = Simulation(args.simulation)
            obj = sim.load_catalog(int(snap))
            snap_path = sim.get_snapshot_file(int(snap))
            extract_particles(obj, snap_path, snap=int(snap), galaxy_id=int(galaxy_id))
            print(f'Completed snap={int(snap)} galaxy_id={int(galaxy_id)}')

        if __name__ == '__main__':
            main()
        '''
    ).strip() + '\n'
 )

slurm_path = job_dir / 'submit_filter_particles_array.sh'
slurm_path.write_text(
    textwrap.dedent(
        f'''
        #!/bin/bash
        #SBATCH --job-name=flt_parts
        #SBATCH --output={log_dir}/%x_%A_%a.out
        #SBATCH --error={log_dir}/%x_%A_%a.err
        #SBATCH --time=04:00:00
        #SBATCH --cpus-per-task=1
        #SBATCH --mem=8G
        #SBATCH --array=0-{len(jobs)-1}

        set -euo pipefail
        cd {Path.cwd()}
        source $HOME/miniconda3/etc/profile.d/conda.sh
        conda activate pd39
        python {worker_path} --manifest {manifest_path} --task-id ${{SLURM_ARRAY_TASK_ID}}
        '''
    ).strip() + '\n'
 )
slurm_path.chmod(0o750)

submit_jobs = False  # flip to True to submit now
if submit_jobs:
    res = subprocess.run(['sbatch', str(slurm_path)], check=True, text=True, capture_output=True)
    print(res.stdout.strip())
else:
    print(f'Prepared {len(jobs)} array tasks.')
    print(f'Manifest: {manifest_path}')
    print(f'SLURM script: {slurm_path}')
    print(f'To submit: sbatch {slurm_path}')

Prepared 487 array tasks.
Manifest: /mnt/home/glorenzon/output/slurm/filter_particles/jobs.tsv
SLURM script: /mnt/home/glorenzon/output/slurm/filter_particles/submit_filter_particles_array.sh
To submit: sbatch /mnt/home/glorenzon/output/slurm/filter_particles/submit_filter_particles_array.sh
